# SearchProbe: Embedding Geometry & Adversarial Analysis

This notebook runs all heavy-computation modules of the SearchProbe research platform on Google Colab's GPU infrastructure.

**Sections:**
1. Setup & Installation
2. Configuration
3. Embedding Geometry Analysis (no API keys needed)
4. Cross-Encoder Validation (needs benchmark data)
5. Perturbation Analysis (needs search provider API key)
6. Adversarial Query Evolution (embedding_sim mode needs no API keys)
7. Export Results

Each section is self-contained. You can run geometry analysis without needing any API keys.

In [ ]:
# =============================================================================
# Cell 1: Setup & Installation
# =============================================================================

# Clone the repository (skip if already cloned or using Drive mount)
import os

REPO_DIR = "searchprobe"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mandipadk/searchprobe.git
os.chdir(REPO_DIR)

# Install with analysis dependencies (sentence-transformers, torch, sklearn, etc.)
!pip install -e ".[analysis]" -q

# Configure API keys via Colab secrets (with fallback to manual input)
def get_secret(name, env_var):
    """Try Colab secrets first, then environment, then prompt."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[env_var] = val
            print(f"  {env_var}: loaded from Colab secrets")
            return
    except (ImportError, Exception):
        pass
    if os.environ.get(env_var):
        print(f"  {env_var}: found in environment")
        return
    print(f"  {env_var}: not set (optional for geometry analysis)")

print("API Key Status:")
get_secret("EXA_API_KEY", "SEARCHPROBE_EXA_API_KEY")
get_secret("TAVILY_API_KEY", "SEARCHPROBE_TAVILY_API_KEY")
get_secret("ANTHROPIC_API_KEY", "SEARCHPROBE_ANTHROPIC_API_KEY")

# Verify GPU availability
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\nSetup complete!")

In [ ]:
# =============================================================================
# Cell 2: Configuration
# =============================================================================

import torch

# --- Models to analyze ---
# Uncomment models you want to include. More models = more comprehensive but slower.
MODELS = [
    "all-MiniLM-L6-v2",          # Fast, lightweight (80MB)
    "all-mpnet-base-v2",          # Higher quality (420MB)
    # "BAAI/bge-large-en-v1.5",   # State-of-the-art (1.2GB) — uncomment for thorough analysis
]

# --- Categories to target (None = all 13) ---
# Set to a list to analyze specific categories only
CATEGORIES = None  # e.g., ["negation", "antonym_confusion", "compositional"]

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Models: {MODELS}")
print(f"Categories: {CATEGORIES or 'all 13'}")

---
## Embedding Geometry Analysis

This section measures **why** embedding-based search fails by analyzing the geometric properties of the embedding space. No API keys needed — only sentence-transformer models.

**Key metrics:**
- **Cosine Similarity**: If `cos(embed("companies in AI"), embed("companies NOT in AI")) = 0.96`, then ANY embedding search will fail on negation
- **Collapse Ratio**: `adversarial_sim / baseline_sim` — ratio > 1 means adversarial pairs are MORE similar than same-topic pairs
- **Vulnerability Score**: Composite 0-1 score per category (higher = more vulnerable)
- **Intrinsic Dimensionality**: How many effective dimensions the embedding space uses for this category
- **Isotropy**: How uniformly distributed embeddings are (1.0 = uniform, ~0 = clustered)

In [ ]:
# =============================================================================
# Cell 3: Run Geometry Analysis
# =============================================================================

from searchprobe.geometry.analyzer import EmbeddingGeometryAnalyzer

analyzer = EmbeddingGeometryAnalyzer(models=MODELS, device=DEVICE)

def progress(model, category, model_idx, cat_idx, total):
    if category:
        print(f"  [{model_idx+1}/{total}] {model} > {category}", end="\r")
    else:
        print(f"\nAnalyzing model {model_idx+1}/{total}: {model}")

print("Running geometry analysis...")
report = analyzer.generate_report(categories=CATEGORIES, progress_callback=progress)
print(f"\nDone! Analyzed {len(report.models)} models x {len(list(report.profiles.values())[0])} categories")

In [ ]:
# =============================================================================
# Cell 4: Vulnerability Heatmap
# =============================================================================

from searchprobe.reporting.charts import create_vulnerability_heatmap, create_similarity_distribution

# Vulnerability heatmap (model x category)
vuln_matrix = report.get_vulnerability_matrix()
fig = create_vulnerability_heatmap(vuln_matrix)
fig.show()

# Print most vulnerable categories per model
for model in report.models:
    top = report.get_most_vulnerable_categories(model, top_n=5)
    print(f"\n{model} — Top 5 vulnerable categories:")
    for cat, score in top:
        severity = "CRITICAL" if score >= 0.8 else "HIGH" if score >= 0.6 else "MODERATE" if score >= 0.4 else "LOW"
        print(f"  {cat:30s} {score:.3f} ({severity})")

In [ ]:
# =============================================================================
# Cell 5: Similarity Distributions
# =============================================================================

# For each model, show the distribution of adversarial vs baseline vs random similarities
for model_name in report.models:
    profiles = report.profiles[model_name]
    all_adv = []
    all_base = []
    all_rand = []
    for profile in profiles.values():
        all_adv.extend(profile.adversarial_similarities)
        all_base.extend(profile.baseline_similarities)
        all_rand.extend(profile.random_similarities)

    fig = create_similarity_distribution(
        all_adv, all_base, all_rand,
        title=f"Similarity Distribution — {model_name}",
    )
    fig.show()

    print(f"\n{model_name}:")
    print(f"  Adversarial mean: {sum(all_adv)/len(all_adv):.3f} (should be LOW)")
    print(f"  Baseline mean:    {sum(all_base)/len(all_base):.3f}")
    print(f"  Random mean:      {sum(all_rand)/len(all_rand):.3f}")
    overlap = sum(1 for a in all_adv if a > min(all_base)) / len(all_adv)
    print(f"  Adversarial-Baseline overlap: {overlap:.1%}")

In [ ]:
# =============================================================================
# Cell 6: Pair-Level Details & 3D Explorer
# =============================================================================

import pandas as pd
from searchprobe.reporting.charts import create_3d_embedding_explorer

# Build pair-level DataFrame for the first model
model_name = report.models[0]
rows = []
for cat, profile in report.profiles[model_name].items():
    for detail in profile.pair_details:
        rows.append({
            "category": cat,
            "query_a": detail["query_a"],
            "query_b": detail["query_b"],
            "cosine_similarity": detail["similarity"],
            "angular_distance": detail["angular_distance"],
        })

df_pairs = pd.DataFrame(rows).sort_values("cosine_similarity", ascending=False)
print(f"Top 15 most similar adversarial pairs ({model_name}):")
print(f"These pairs SHOULD be different but the embedding model collapses them.\n")
display(df_pairs.head(15))

# 3D Embedding Explorer — project all adversarial queries into 3D space
# Collect all unique queries and their categories
all_texts = []
all_cats = []
seen = set()
for cat, profile in report.profiles[model_name].items():
    for detail in profile.pair_details:
        for q in [detail["query_a"], detail["query_b"]]:
            if q not in seen:
                all_texts.append(q)
                all_cats.append(cat)
                seen.add(q)

# Encode and reduce to 3D
embeddings = analyzer._encode(model_name, all_texts)

try:
    from umap import UMAP
    reducer = UMAP(n_components=3, random_state=42, n_neighbors=min(15, len(all_texts)-1))
    method = "UMAP"
except ImportError:
    from sklearn.manifold import TSNE
    reducer = TSNE(n_components=3, random_state=42, perplexity=min(30, len(all_texts)-1))
    method = "t-SNE"

coords_3d = reducer.fit_transform(embeddings)
embeddings_3d = [(float(c[0]), float(c[1]), float(c[2])) for c in coords_3d]

fig = create_3d_embedding_explorer(embeddings_3d, all_texts, all_cats,
                                    title=f"Adversarial Query Embedding Space — {model_name}")
fig.show()

---
## Cross-Encoder Validation

Cross-encoders jointly encode (query, document) pairs at O(n) cost, producing much more accurate scores than bi-encoders. By re-scoring search results, we quantify the **embedding gap** — how much relevance quality is lost by using bi-encoder approximation.

**Requires:** Either a prior benchmark run (searchprobe.db), or we can run a small demo with hardcoded results below.

In [ ]:
# =============================================================================
# Cell 7: Cross-Encoder Setup & Demo Validation
# =============================================================================

from searchprobe.validation.cross_encoder import CrossEncoderValidator

validator = CrossEncoderValidator(device=DEVICE)

# Demo: validate with sample search results
# In production, these would come from a benchmark run against real search providers
demo_queries = [
    {
        "query_id": "demo_neg_1",
        "query_text": "companies that are NOT in artificial intelligence",
        "category": "negation",
        "provider": "demo",
        "results": [
            {"title": "Top AI Companies 2024", "url": "https://example.com/1",
             "snippet": "Leading artificial intelligence companies including OpenAI, Google DeepMind..."},
            {"title": "AI Startups to Watch", "url": "https://example.com/2",
             "snippet": "Emerging AI startups disrupting industries with machine learning..."},
            {"title": "Traditional Manufacturing Companies", "url": "https://example.com/3",
             "snippet": "Steel and automotive manufacturing companies with no AI adoption..."},
            {"title": "Non-Tech Fortune 500", "url": "https://example.com/4",
             "snippet": "Fortune 500 companies in retail, energy, and logistics sectors..."},
            {"title": "Enterprise AI Solutions", "url": "https://example.com/5",
             "snippet": "Enterprise software companies building AI-powered solutions..."},
        ],
    },
    {
        "query_id": "demo_comp_1",
        "query_text": "startups that acquired large companies",
        "category": "compositional",
        "provider": "demo",
        "results": [
            {"title": "Google Acquires Startup", "url": "https://example.com/6",
             "snippet": "Google acquires AI startup for $2 billion..."},
            {"title": "Startup Acquisitions in Tech", "url": "https://example.com/7",
             "snippet": "Small startups that grew to acquire larger competitors..."},
            {"title": "M&A Trends 2024", "url": "https://example.com/8",
             "snippet": "Merger and acquisition trends in the technology sector..."},
        ],
    },
]

# Run validation
validation_results = []
for item in demo_queries:
    result = validator.validate_search_results(
        query_id=item["query_id"],
        query_text=item["query_text"],
        category=item["category"],
        provider=item["provider"],
        results=item["results"],
    )
    validation_results.append(result)
    print(f"Query: {item['query_text'][:60]}...")
    print(f"  Category: {item['category']}")
    print(f"  Original NDCG:  {result.original_ndcg:.3f}")
    print(f"  Reranked NDCG:  {result.reranked_ndcg:.3f}")
    print(f"  NDCG Improvement: {result.ndcg_improvement:.3f}")
    print(f"  Kendall's tau:  {result.kendall_tau:.3f}")
    print()

In [ ]:
# =============================================================================
# Cell 8: Embedding Gap Analysis
# =============================================================================

from searchprobe.validation.gap_analysis import EmbeddingGapAnalyzer
from searchprobe.reporting.charts import create_embedding_gap_chart

gap_analyzer = EmbeddingGapAnalyzer()

# Get category ranking by gap severity
category_ranking = gap_analyzer.get_category_ranking(validation_results)
print("Category Ranking by Embedding Gap:")
for cat, improvement in category_ranking:
    severity = "SEVERE" if improvement >= 0.3 else "SIGNIFICANT" if improvement >= 0.15 else "MODERATE" if improvement >= 0.05 else "MINIMAL"
    print(f"  {cat:30s} NDCG improvement: {improvement:.3f} ({severity})")

# Visualize
gap_data = {cat: imp for cat, imp in category_ranking}
fig = create_embedding_gap_chart(gap_data)
fig.show()

In [ ]:
# =============================================================================
# Cell 9: Per-Result Cross-Encoder Scores
# =============================================================================

import pandas as pd

# Show detailed cross-encoder scores for each query
for vr in validation_results:
    print(f"\nQuery: {vr.query_text}")
    print(f"Category: {vr.category} | NDCG improvement: {vr.ndcg_improvement:.3f}")
    rows = []
    for score in vr.scores:
        rows.append({
            "title": score.document_title[:50],
            "original_rank": score.original_rank,
            "ce_score": f"{score.cross_encoder_score:.4f}",
            "reranked_pos": score.reranked_position,
            "rank_change": score.original_rank - score.reranked_position,
        })
    display(pd.DataFrame(rows))

---
## Perturbation Analysis

Systematically perturbs queries (word deletion, word swap, negation insertion, synonym replacement) and measures how much search results change. This produces **sensitivity maps** showing which words are "load-bearing" for retrieval.

**Requires:** A search provider API key (e.g., Exa). The perturbation operators themselves work locally, but testing result stability requires live search.

In [ ]:
# =============================================================================
# Cell 10: Perturbation Operators (Local Demo)
# =============================================================================

from searchprobe.perturbation.operators import apply_perturbation, PerturbationType

# Demonstrate perturbation operators on sample queries
sample_queries = [
    "companies that are NOT in artificial intelligence",
    "startups with more than 50 employees founded after 2020",
    "Python libraries for computer vision released recently",
]

for query in sample_queries:
    print(f"\nOriginal: {query}")
    print("-" * 60)
    for op_type in PerturbationType:
        variants = apply_perturbation(query, op_type, max_variants=2)
        if variants:
            print(f"  {op_type.value}:")
            for variant, detail in variants[:2]:
                print(f"    -> {variant}  ({detail})")

In [ ]:
# =============================================================================
# Cell 11: Live Perturbation Analysis (Requires Search Provider API Key)
# =============================================================================

# Uncomment and configure to run live perturbation analysis
# This requires a search provider API key set in Cell 1

import asyncio
import os

HAS_EXA_KEY = bool(os.environ.get("SEARCHPROBE_EXA_API_KEY"))

if HAS_EXA_KEY:
    from searchprobe.providers.exa import ExaProvider
    from searchprobe.perturbation.engine import PerturbationEngine
    from searchprobe.perturbation.operators import PerturbationType

    provider = ExaProvider(api_key=os.environ["SEARCHPROBE_EXA_API_KEY"])
    engine = PerturbationEngine(
        provider=provider,
        operators=[PerturbationType.WORD_DELETE, PerturbationType.WORD_SWAP, PerturbationType.NEGATION_INSERT],
        max_variants_per_operator=3,
    )

    queries = [
        {"query": "companies that are NOT in artificial intelligence", "category": "negation"},
        {"query": "startups with more than 50 employees", "category": "numeric_precision"},
    ]

    perturb_report = await engine.analyze_queries(queries)

    print(f"Perturbation Report:")
    print(f"  Mean Jaccard Similarity: {perturb_report.mean_jaccard:.3f}")
    print(f"  Mean RBO Score: {perturb_report.mean_rbo:.3f}")
    print(f"\nStability by Operator:")
    for op, score in perturb_report.stability_by_operator.items():
        print(f"  {op:25s} {score:.3f}")
else:
    print("Skipping live perturbation analysis (no SEARCHPROBE_EXA_API_KEY set).")
    print("Set your API key in Colab secrets or Cell 1 to enable this section.")
    perturb_report = None

In [ ]:
# =============================================================================
# Cell 12: Sensitivity Map Visualization
# =============================================================================

from searchprobe.reporting.charts import create_sensitivity_map_chart

if perturb_report and perturb_report.sensitivity_maps:
    for smap in perturb_report.sensitivity_maps[:5]:  # Show up to 5 maps
        fig = create_sensitivity_map_chart(smap.query, smap.word_scores,
                                          title=f"Sensitivity Map ({smap.category})")
        fig.show()

        print(f"\nQuery: {smap.query}")
        print(f"Most load-bearing words:")
        for word, score in smap.get_most_sensitive_words(5):
            bar = '#' * int(score * 20)
            print(f"  {word:20s} {score:.3f} {bar}")
else:
    # Demo with synthetic sensitivity data
    demo_query = "companies that are NOT in artificial intelligence"
    demo_scores = {
        "companies": 0.3, "that": 0.1, "are": 0.05,
        "NOT": 0.95, "in": 0.15, "artificial": 0.6, "intelligence": 0.7,
    }
    fig = create_sensitivity_map_chart(demo_query, demo_scores,
                                      title="Sensitivity Map (Demo — Negation)")
    fig.show()
    print("Showing demo sensitivity map. Run Cell 11 with an API key for live results.")

---
## Adversarial Query Evolution

Uses evolutionary optimization to breed worst-case adversarial queries. Instead of hand-crafting adversarial queries, this discovers failure modes humans wouldn't think of.

**Fitness modes:**
- `embedding_sim` (default) — Uses structural heuristics, **no API keys needed**
- `cross_encoder` — Uses cross-encoder gap, needs search provider API key
- `llm_judge` — Most accurate, needs search provider + LLM API keys

In [ ]:
# =============================================================================
# Cell 13: Configure Evolution
# =============================================================================

from searchprobe.adversarial.models import EvolutionConfig
from searchprobe.adversarial.fitness import FitnessEvaluator
from searchprobe.adversarial.optimizer import AdversarialQueryOptimizer

# Seed queries to start evolution from
SEED_QUERIES = [
    "companies that are NOT in artificial intelligence",
    "startups with exactly 50 employees founded before 2020",
    "Python web frameworks that are NOT Django and NOT Flask",
    "research papers published between January and March 2024 only",
    "foods that contain gluten but are not bread or pasta",
    "cities with population under 100000 that have international airports",
    "open source databases written in Rust with ACID transactions",
    "companies that failed despite having good products",
]

config = EvolutionConfig(
    population_size=20,      # Smaller for demo; use 30-50 for thorough runs
    generations=10,          # Fewer for demo; use 20-50 for thorough runs
    mutation_rate=0.7,
    crossover_rate=0.3,
    elitism_count=3,
    tournament_size=3,
    fitness_mode="embedding_sim",  # No API keys needed
    seed_queries=SEED_QUERIES,
    target_categories=["negation", "numeric_precision", "multi_constraint", "boolean_logic"],
)

fitness = FitnessEvaluator(mode=config.fitness_mode)
optimizer = AdversarialQueryOptimizer(config=config, fitness_evaluator=fitness)

print(f"Evolution config:")
print(f"  Population: {config.population_size}")
print(f"  Generations: {config.generations}")
print(f"  Fitness mode: {config.fitness_mode}")
print(f"  Seeds: {len(SEED_QUERIES)}")

In [ ]:
# =============================================================================
# Cell 14: Run Evolution
# =============================================================================

import asyncio

def evolution_progress(gen, total, best_fit, mean_fit):
    bar = '#' * int(best_fit * 30)
    print(f"  Gen {gen+1:3d}/{total} | Best: {best_fit:.3f} | Mean: {mean_fit:.3f} | {bar}", end="\r")

print("Running adversarial query evolution...")
opt_result = await optimizer.optimize(progress_callback=evolution_progress)
print(f"\n\nEvolution complete!")
print(f"  Generations: {opt_result.generations_completed}")
print(f"  Total evaluations: {opt_result.total_evaluations}")

print(f"\nTop 10 evolved adversarial queries:")
for i, ind in enumerate(opt_result.best_individuals[:10]):
    print(f"  {i+1}. [{ind.fitness:.3f}] {ind.query}")
    if ind.mutation_history:
        print(f"     Mutations: {' -> '.join(ind.mutation_history[-3:])}")

In [ ]:
# =============================================================================
# Cell 15: Evolution Fitness History
# =============================================================================

import plotly.graph_objects as go

# Plot fitness over generations
gens = [h["generation"] for h in opt_result.fitness_history]
means = [h["mean"] for h in opt_result.fitness_history]
maxes = [h["max"] for h in opt_result.fitness_history]
mins = [h["min"] for h in opt_result.fitness_history]

fig = go.Figure()
fig.add_trace(go.Scatter(x=gens, y=maxes, name="Best", line=dict(color="#e74c3c", width=2)))
fig.add_trace(go.Scatter(x=gens, y=means, name="Mean", line=dict(color="#3498db", width=2)))
fig.add_trace(go.Scatter(x=gens, y=mins, name="Worst", line=dict(color="#95a5a6", width=1, dash="dash")))
fig.update_layout(
    title="Adversarial Fitness Over Generations",
    xaxis_title="Generation",
    yaxis_title="Fitness Score",
    yaxis=dict(range=[0, 1]),
    height=400,
    width=700,
)
fig.show()

# Analyze mutation effectiveness
mutation_counts = {}
for ind in opt_result.best_individuals:
    for mut in ind.mutation_history:
        mutation_counts[mut] = mutation_counts.get(mut, 0) + 1

if mutation_counts:
    print("\nMost common mutations in top individuals:")
    for mut, count in sorted(mutation_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {mut:30s} x{count}")

In [ ]:
# =============================================================================
# Cell 16: Export Results
# =============================================================================

import json

# Export geometry report
geometry_export = {
    "models": report.models,
    "generated_at": report.generated_at.isoformat(),
    "profiles": {
        model: {
            cat: profile.to_dict()
            for cat, profile in categories.items()
        }
        for model, categories in report.profiles.items()
    },
    "vulnerability_matrix": report.get_vulnerability_matrix(),
}

with open("geometry_report.json", "w") as f:
    json.dump(geometry_export, f, indent=2)
print("Saved: geometry_report.json")

# Export validation results
if validation_results:
    validation_export = [vr.to_dict() for vr in validation_results]
    with open("validation_results.json", "w") as f:
        json.dump(validation_export, f, indent=2)
    print("Saved: validation_results.json")

# Export evolution results
if opt_result:
    evolution_export = opt_result.to_dict()
    with open("evolution_results.json", "w") as f:
        json.dump(evolution_export, f, indent=2)
    print("Saved: evolution_results.json")

# Download files (Colab only)
try:
    from google.colab import files
    for fname in ["geometry_report.json", "validation_results.json", "evolution_results.json"]:
        if os.path.exists(fname):
            files.download(fname)
except ImportError:
    print("\nNot running in Colab — files saved to current directory.")